[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shreyashgol/phonetic-guided-speech-generation/blob/main/notebooks/17_demo_app.ipynb)

# 17 · Demo app: hear what explicit durations do
**Goal:** a live web app (Gradio) to show the whole system to the judges: type a sentence, hear it, **see and edit the phoneme durations**, and compare duration sources.

**Use a GPU:** Runtime -> Change runtime type -> T4. (a CPU also works, only slower). Run **after** notebooks 10–13 (data contract, trained checkpoint `best.pt`).

**What the app does**
- **Speak:** text → phonemes → durations → mel → speech, with a duration plot and a mel plot with phoneme boundaries.
- **Controls:** duration source, speed (duration scale), pause length.
- **Edit durations:** the phoneme table is editable; change a duration (or a phoneme) and press *Re-synthesize*. This shows the point of explicit durations: you can *control* timing.
- **Compare duration sources:** the same sentence with the model's duration head, the phone-mean baseline and the global-mean baseline, side by side. Only the durations change: the Phase 2 experiment in miniature.

**Design decisions (put these in the report/viva)**
- The engine (compute) and the UI are separate modules, so the same engine can be used from a script, a notebook or a hosted app.
- The Phase 1 BiLSTM is evaluated offline in notebook 14; for new text it needs the Phase 1 feature builder, which is listed as future work. The app says so on its *About* tab.

**Writes:** `src/app/engine.py`, `src/app/gradio_app.py`, `app.py`, `requirements_demo.txt`, `README_DEMO.md`.

In [1]:
# ---- Cell 0: Setup (identical at the top of every notebook)
import os, sys, json, glob
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/phonetic-tts'
    IN_COLAB = True
except ImportError:                      # running somewhere else (local / tests)
    PROJECT_ROOT = os.environ.get('PROJECT_ROOT', os.getcwd())
    IN_COLAB = False
sys.path.insert(0, PROJECT_ROOT)
from src.utils.config import load_config
cfg = load_config(PROJECT_ROOT)
A = cfg['audio']; P = cfg['abs_paths']
RAW, PROCESSED, ALIGN, SPLITS, RESULTS = P['raw'], P['processed'], P['alignments'], P['splits'], P['results']
for d in (PROCESSED, ALIGN, SPLITS, RESULTS):
    os.makedirs(d, exist_ok=True)
print('Project root:', PROJECT_ROOT, '| Colab:', IN_COLAB)

Mounted at /content/drive
Project root: /content/drive/MyDrive/phonetic-tts | Colab: True


In [2]:
%%bash
pip -q install gradio speechbrain g2p_en

In [3]:
# ---- Write shared module: acoustic model
import os, importlib
_src = r'''"""FastSpeech 2-style acoustic model (phonemes + durations -> mel). No pitch/energy predictors (added later as an ablation)."""
import math
import torch, torch.nn as nn, torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence


def sinusoid(n, d):
    pos = torch.arange(n).unsqueeze(1).float()
    ang = pos / torch.pow(10000.0, torch.arange(0, d, 2).float() / d)
    pe = torch.zeros(n, d); pe[:, 0::2] = torch.sin(ang); pe[:, 1::2] = torch.cos(ang)
    return pe


class FFTBlock(nn.Module):
    """Self-attention + 1D-conv feed-forward, post-LayerNorm (as in FastSpeech)."""
    def __init__(self, d, heads, ffn, k, p):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, dropout=p, batch_first=True)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.c1 = nn.Conv1d(d, ffn, k, padding=k // 2); self.c2 = nn.Conv1d(ffn, d, 1)
        self.drop = nn.Dropout(p)

    def forward(self, x, pad):                       # x (B,T,D); pad (B,T) True = padding
        a, _ = self.attn(x, x, x, key_padding_mask=pad, need_weights=False)
        x = self.n1(x + self.drop(a))
        y = x.masked_fill(pad.unsqueeze(-1), 0).transpose(1, 2)
        y = self.c2(F.relu(self.c1(y))).transpose(1, 2)
        x = self.n2(x + self.drop(y))
        return x.masked_fill(pad.unsqueeze(-1), 0)


class DurationPredictor(nn.Module):
    """Predicts log(frames + 1) per token (the model's own duration head)."""
    def __init__(self, d, k=3, p=0.2):
        super().__init__()
        self.c1 = nn.Conv1d(d, d, k, padding=k // 2); self.n1 = nn.LayerNorm(d)
        self.c2 = nn.Conv1d(d, d, k, padding=k // 2); self.n2 = nn.LayerNorm(d)
        self.out = nn.Linear(d, 1); self.drop = nn.Dropout(p)

    def forward(self, x, pad):
        m = pad.unsqueeze(-1)
        y = self.drop(self.n1(F.relu(self.c1(x.masked_fill(m, 0).transpose(1, 2))).transpose(1, 2)))
        y = self.drop(self.n2(F.relu(self.c2(y.masked_fill(m, 0).transpose(1, 2))).transpose(1, 2)))
        return self.out(y).squeeze(-1).masked_fill(pad, 0)


def length_regulate(x, dur):
    """Repeat each token vector dur[b, t] times. x (B,T,D), dur (B,T) long -> (B,M,D), lengths (B,)"""
    outs = [torch.repeat_interleave(x[b], dur[b], dim=0) for b in range(x.size(0))]
    lens = torch.tensor([o.size(0) for o in outs], device=x.device)
    return pad_sequence(outs, batch_first=True), lens


class AcousticModel(nn.Module):
    def __init__(self, n_vocab, n_mels=80, d=256, heads=2, ffn=768, k=3, n_enc=4, n_dec=4, p=0.1, max_len=1600):
        super().__init__()
        self.tok = nn.Embedding(n_vocab, d, padding_idx=0); self.stress = nn.Embedding(4, d)
        self.register_buffer('pe', sinusoid(max_len, d), persistent=False)
        self.enc = nn.ModuleList([FFTBlock(d, heads, ffn, k, p) for _ in range(n_enc)])
        self.dec = nn.ModuleList([FFTBlock(d, heads, ffn, k, p) for _ in range(n_dec)])
        self.dp = DurationPredictor(d); self.out = nn.Linear(d, n_mels)

    def forward(self, tok, stress, dur=None, alpha=1.0):
        """dur given -> teacher forcing (training / ground truth / Phase 1 durations). dur None -> own duration head."""
        pad = tok == 0
        x = self.tok(tok) + self.stress(stress) + self.pe[:tok.size(1)]
        for b in self.enc: x = b(x, pad)
        logd = self.dp(x, pad)
        if dur is None:
            dur = torch.clamp(torch.round((torch.exp(logd) - 1) * alpha), min=1).long().masked_fill(pad, 0)
        y, lens = length_regulate(x, dur)
        mpad = torch.arange(y.size(1), device=y.device)[None] >= lens[:, None]
        y = (y + self.pe[:y.size(1)]).masked_fill(mpad.unsqueeze(-1), 0)
        for b in self.dec: y = b(y, mpad)
        return dict(mel=self.out(y), logd=logd, tok_pad=pad, mel_pad=mpad, dur=dur)


def acoustic_loss(o, mel, dur):
    """masked L1 on mel + masked MSE on log-duration."""
    assert o['mel'].shape == mel.shape, (o['mel'].shape, mel.shape)
    m = (~o['mel_pad']).unsqueeze(-1).float()
    mel_l = ((o['mel'] - mel).abs() * m).sum() / (m.sum() * mel.size(-1))
    t = (~o['tok_pad']).float()
    dur_l = (((o['logd'] - torch.log(dur.float() + 1)) ** 2) * t).sum() / t.sum()
    return mel_l + dur_l, mel_l, dur_l
'''
_p = os.path.join(PROJECT_ROOT, "src/models/acoustic/model.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/models/acoustic/model.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/models/acoustic/model.py


In [4]:
# ---- Write shared module: pretrained HiFi-GAN wrapper
import os, importlib
_src = r'''"""Pretrained HiFi-GAN (SpeechBrain, trained on LJSpeech at 22.05 kHz, 80 mels, hop 256)."""
import torch


class Vocoder:
    def __init__(self, device='cpu', savedir='pretrained_hifigan'):
        try:
            from speechbrain.inference.vocoders import HIFIGAN
        except ImportError:                                   # older speechbrain
            from speechbrain.pretrained import HIFIGAN
        self.device = device
        self.m = HIFIGAN.from_hparams(source='speechbrain/tts-hifigan-ljspeech', savedir=savedir, run_opts={'device': device})

    @torch.no_grad()
    def __call__(self, mel):
        """mel: (B, T, 80) or (T, 80) log-mel in natural units (NOT normalised) -> waveform (B, T*256) float tensor on CPU."""
        if mel.dim() == 2: mel = mel.unsqueeze(0)
        w = self.m.decode_batch(mel.transpose(1, 2).float().to(self.device))
        return w.reshape(w.shape[0], -1).cpu()
'''
_p = os.path.join(PROJECT_ROOT, "src/vocoder/hifigan.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/vocoder/hifigan.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/vocoder/hifigan.py


In [5]:
# ---- Write shared module: demo engine (compute only)
import os, importlib
_src = r'''"""Demo engine: text -> phonemes -> durations -> mel -> wav. Pure compute (no UI). Used by the Gradio app and reusable elsewhere."""
import json, os, re
import numpy as np, pandas as pd, torch
from src.models.acoustic.model import AcousticModel
from src.vocoder.hifigan import Vocoder

SOURCES = {'model': 'Acoustic model duration head', 'phone_mean': 'Phone-mean baseline', 'global_mean': 'Global-mean baseline'}


class TTSEngine:
    def __init__(self, processed_dir, ckpt_path, hifigan_dir, device=None, sr=22050, hop=256):
        from g2p_en import G2p
        self.dev = device or ('cuda' if torch.cuda.is_available() else 'cpu'); self.sr, self.hop = sr, hop
        c = json.load(open(os.path.join(processed_dir, 'phase2_contract.json')))
        self.vocab = c['vocab']; self.id2ph = {v: k for k, v in self.vocab.items()}; self.SIL = self.vocab['sil']
        self.sil = json.load(open(os.path.join(processed_dir, 'sil_stats.json')))
        st = json.load(open(os.path.join(processed_dir, 'mel_stats.json')))
        self.mean = np.array(st['mean'], np.float32); self.std = np.array(st['std'], np.float32) + 1e-5
        ck = torch.load(ckpt_path, map_location=self.dev, weights_only=False)
        self.model = AcousticModel(len(self.vocab), len(self.mean), **ck['arch']).to(self.dev); self.model.load_state_dict(ck['model']); self.model.eval()
        self.step = int(ck['step']); self.voc = Vocoder(self.dev, hifigan_dir); self.g2p = G2p()
        man = pd.read_csv(os.path.join(processed_dir, 'phase2_manifest.csv')); man = man[man.split == 'train']
        tk = np.concatenate([np.array(s.split(), int) for s in man.tokens]); du = np.concatenate([np.array(s.split(), int) for s in man.durations])
        sp = tk != self.SIL
        self.phone_median = {int(t): float(np.median(du[tk == t])) for t in np.unique(tk[sp])}; self.global_median = float(np.median(du[sp]))

    # ---- text -> tokens
    def label(self, tok, stress):
        b = self.id2ph[int(tok)]
        return 'sil' if int(tok) == self.SIL else (b + (str(int(stress) - 1) if int(stress) > 0 else ''))

    def text_to_tokens(self, text):
        toks, stress = [self.SIL], [0]
        for p in self.g2p(text.lower()):
            if p == ' ': continue
            if p in ',;:.!?':
                if toks[-1] != self.SIL: toks.append(self.SIL); stress.append(0)
                continue
            m = re.fullmatch(r'([A-Z]+)([012])?', p)
            if not m or m.group(1) not in self.vocab: continue
            toks.append(self.vocab[m.group(1)]); stress.append(0 if m.group(2) is None else int(m.group(2)) + 1)
        if toks[-1] != self.SIL: toks.append(self.SIL); stress.append(0)
        return np.array(toks), np.array(stress)

    # ---- durations
    def durations(self, tok, stress, source='model', alpha=1.0, pause_scale=1.0):
        b = lambda a: torch.from_numpy(a)[None].to(self.dev)
        n = len(tok)
        if source == 'model':
            with torch.no_grad(): d = self.model(b(tok), b(stress), alpha=alpha)['dur'][0].cpu().numpy().astype(float)
        elif source == 'phone_mean':
            d = np.array([self.phone_median.get(int(t), self.global_median) for t in tok]) * alpha
        elif source == 'global_mean':
            d = np.full(n, self.global_median) * alpha
        else: raise ValueError(source)
        is_sil = tok == self.SIL; pos = np.arange(n)
        d[is_sil] = self.sil['internal']; d[is_sil & (pos == 0)] = self.sil['leading']; d[is_sil & (pos == n - 1)] = self.sil['trailing']
        d[is_sil] *= pause_scale
        return np.maximum(np.round(d), 1).astype(np.int64)

    def table(self, tok, stress, dur):
        return pd.DataFrame({'phone': [self.label(t, s) for t, s in zip(tok, stress)], 'duration_frames': dur.astype(int)})

    def tokens_from_table(self, df):
        tok, stress, dur = [], [], []
        for p, d in zip(df['phone'], df['duration_frames']):
            p = str(p).strip()
            if not p or pd.isna(d): continue
            if p.lower() == 'sil': t, s = self.SIL, 0
            else:
                m = re.fullmatch(r'([A-Za-z]+)([012])?', p)
                if not m or m.group(1).upper() not in self.vocab: raise ValueError(f'Unknown phone: {p}')
                t, s = self.vocab[m.group(1).upper()], (0 if m.group(2) is None else int(m.group(2)) + 1)
            tok.append(t); stress.append(s); dur.append(max(1, int(round(float(d)))))
        if not tok: raise ValueError('The table is empty.')
        return np.array(tok), np.array(stress), np.array(dur, np.int64)

    # ---- synthesis
    @torch.no_grad()
    def synthesize(self, tok, stress, dur):
        b = lambda a: torch.from_numpy(a)[None].to(self.dev)
        mel = self.model(b(tok), b(stress), dur=b(dur))['mel'][0].cpu().numpy() * self.std + self.mean
        wav = np.clip(self.voc(torch.from_numpy(mel.astype(np.float32)))[0].numpy(), -1, 1).astype(np.float32)
        return wav, mel

    def _pack(self, tok, stress, dur):
        wav, mel = self.synthesize(tok, stress, dur)
        return dict(wav=wav, mel=mel, sr=self.sr, table=self.table(tok, stress, dur), seconds=float(dur.sum() * self.hop / self.sr))

    def speak(self, text, source='model', alpha=1.0, pause_scale=1.0):
        tok, stress = self.text_to_tokens(text)
        if (tok != self.SIL).sum() == 0: raise ValueError('No pronounceable English words found.')
        return self._pack(tok, stress, self.durations(tok, stress, source, alpha, pause_scale))

    def speak_from_table(self, df):
        tok, stress, dur = self.tokens_from_table(df)
        return self._pack(tok, stress, dur)

    def compare(self, text, alpha=1.0, pause_scale=1.0):
        tok, stress = self.text_to_tokens(text)
        if (tok != self.SIL).sum() == 0: raise ValueError('No pronounceable English words found.')
        return {s: self._pack(tok, stress, self.durations(tok, stress, s, alpha, pause_scale)) for s in SOURCES}
'''
_p = os.path.join(PROJECT_ROOT, "src/app/engine.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/app/engine.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/app/engine.py


In [6]:
# ---- Write shared module: Gradio user interface
import os, importlib
_src = r'''"""Gradio UI for the demo. Needs only an engine object with speak / speak_from_table / compare (see engine.py)."""
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr

SOURCE_CHOICES = [('Acoustic model duration head', 'model'), ('Phone-mean baseline', 'phone_mean'), ('Global-mean baseline', 'global_mean')]
EXAMPLES = ['Phonetic guided speech generation uses explicit phoneme durations.',
            'The quick brown fox jumps over the lazy dog, and then it runs away.',
            'Duration control is a direct benefit of predicting phoneme lengths.']
VOWELS = set('AA AE AH AO AW AY EH ER EY IH IY OW OY UH UW'.split())


def _color(label):
    b = label.rstrip('012')
    return '#9aa5b1' if b.lower() == 'sil' else ('#2a7ab0' if b in VOWELS else '#e08a2e')


def fig_durations(tbl):
    f, ax = plt.subplots(figsize=(max(7, 0.32 * len(tbl)), 3))
    ax.bar(range(len(tbl)), tbl.duration_frames, color=[_color(p) for p in tbl.phone])
    ax.set_xticks(range(len(tbl))); ax.set_xticklabels(tbl.phone, rotation=90, fontsize=8)
    ax.set_ylabel('frames (11.6 ms)'); ax.set_title('Phoneme durations (blue vowels, orange consonants, grey silence)')
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
    f.tight_layout(); return f


def fig_mel(mel, tbl):
    f, ax = plt.subplots(figsize=(10, 3)); ax.imshow(mel.T, origin='lower', aspect='auto', cmap='magma')
    edges = np.cumsum(tbl.duration_frames.to_numpy())
    for e in edges[:-1]: ax.axvline(e, color='w', lw=.3, alpha=.5)
    for s0, d, p in zip(edges - tbl.duration_frames.to_numpy(), tbl.duration_frames.to_numpy(), tbl.phone):
        if d >= 4 and p != 'sil': ax.text(s0 + d / 2, mel.shape[1] - 5, p, color='cyan', ha='center', fontsize=7)
    ax.set_xlabel('frame'); ax.set_ylabel('mel bin'); ax.set_title('Generated mel spectrogram with phoneme boundaries'); f.tight_layout(); return f


def fig_compare(res):
    names = {'model': 'model head', 'phone_mean': 'phone mean', 'global_mean': 'global mean'}
    base = res['model']['table']; f, ax = plt.subplots(figsize=(max(7, 0.4 * len(base)), 3.2)); w = 0.27
    for k, (s, r) in enumerate(res.items()):
        ax.bar(np.arange(len(base)) + (k - 1) * w, r['table'].duration_frames, width=w, label=names[s])
    ax.set_xticks(range(len(base))); ax.set_xticklabels(base.phone, rotation=90, fontsize=8); ax.legend(frameon=False); ax.set_ylabel('frames')
    ax.set_title('Predicted duration of each phoneme by source');
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
    f.tight_layout(); return f


def build_demo(engine):
    def outs(r): return (r['sr'], r['wav']), fig_durations(r['table']), fig_mel(r['mel'], r['table']), r['table'], f"Length: {r['seconds']:.2f} s | {len(r['table'])} tokens"

    def gen(text, source, alpha, pause):
        try: return outs(engine.speak(text, source, alpha, pause))
        except Exception as e: raise gr.Error(str(e))

    def resynth(df):
        try: r = engine.speak_from_table(pd.DataFrame(df, columns=['phone', 'duration_frames']) if not isinstance(df, pd.DataFrame) else df)
        except Exception as e: raise gr.Error(str(e))
        o = outs(r); return o[0], o[1], o[2], o[4]

    def cmp(text, alpha):
        try: res = engine.compare(text, alpha)
        except Exception as e: raise gr.Error(str(e))
        info = ' | '.join(f"{n}: {res[k]['seconds']:.2f} s" for k, n in [('model', 'model head'), ('phone_mean', 'phone mean'), ('global_mean', 'global mean')])
        return (res['model']['sr'], res['model']['wav']), (res['phone_mean']['sr'], res['phone_mean']['wav']), (res['global_mean']['sr'], res['global_mean']['wav']), fig_compare(res), info

    with gr.Blocks(title='Phonetic Guided Speech Generation') as demo:
        gr.Markdown('# Phonetic Guided Speech Generation\nText → phonemes → **explicit durations** → mel spectrogram → HiFi-GAN → speech. Change the durations and hear what changes.')
        with gr.Tab('Speak'):
            txt = gr.Textbox(value=EXAMPLES[0], label='Text (English)', lines=2)
            with gr.Row():
                src = gr.Dropdown(choices=SOURCE_CHOICES, value='model', label='Duration source')
                speed = gr.Slider(0.6, 1.6, value=1.0, step=0.05, label='Duration scale (higher = slower speech)')
                pause = gr.Slider(0.0, 2.5, value=1.0, step=0.1, label='Pause length scale')
            btn = gr.Button('Generate', variant='primary'); gr.Examples(EXAMPLES, inputs=txt)
            info = gr.Markdown(); audio = gr.Audio(label='Speech', type='numpy')
            with gr.Row(): pd_plot = gr.Plot(label='Durations'); mel_plot = gr.Plot(label='Mel')
            tbl = gr.Dataframe(headers=['phone', 'duration_frames'], datatype=['str', 'number'], interactive=True, label='Phonemes and durations: edit any cell, then press Re-synthesize')
            btn2 = gr.Button('Re-synthesize from table')
            btn.click(gen, [txt, src, speed, pause], [audio, pd_plot, mel_plot, tbl, info])
            btn2.click(resynth, [tbl], [audio, pd_plot, mel_plot, info])
        with gr.Tab('Compare duration sources'):
            gr.Markdown('Same sentence, same acoustic model; **only the durations change**. This is the Phase 2 experiment in miniature.')
            ctxt = gr.Textbox(value=EXAMPLES[1], label='Text', lines=2); cs = gr.Slider(0.6, 1.6, value=1.0, step=0.05, label='Duration scale')
            cb = gr.Button('Compare', variant='primary'); cinfo = gr.Markdown()
            with gr.Row(): a1 = gr.Audio(label='Model duration head', type='numpy'); a2 = gr.Audio(label='Phone-mean baseline', type='numpy'); a3 = gr.Audio(label='Global-mean baseline', type='numpy')
            cplot = gr.Plot(); cb.click(cmp, [ctxt, cs], [a1, a2, a3, cplot, cinfo])
        with gr.Tab('About'):
            gr.Markdown(f"""
**Pipeline.** g2p_en (text → ARPAbet) → FastSpeech 2-style acoustic model (no pitch/energy) → pretrained HiFi-GAN vocoder.
**Phase 1 link.** The model was trained on MFA forced-alignment durations; the evaluation compares durations from the Phase 1 BiLSTM, simple baselines and the model's own head on the test set.
**In this demo** the duration source for free text is the model's own head or a baseline. The Phase 1 BiLSTM is evaluated offline (it needs the Phase 1 feature builder for new text, which is future work).
**Limits.** Single speaker (LJSpeech), small model, no pitch or energy control, English only.
""")
    return demo
'''
_p = os.path.join(PROJECT_ROOT, "src/app/gradio_app.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/app/gradio_app.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/app/gradio_app.py


## Step 1: Load the engine and run a smoke test (before showing anyone)

In [9]:
import time, numpy as np
import nltk
nltk.download('averaged_perceptron_tagger_eng')

from src.app.engine import TTSEngine
from src.app.gradio_app import build_demo
engine = TTSEngine(PROCESSED, os.path.join(PROJECT_ROOT, 'checkpoints', 'acoustic', 'best.pt'), os.path.join(PROJECT_ROOT, 'pretrained', 'hifigan'))
print('device', engine.dev, '| checkpoint step', engine.step, '| global median duration', engine.global_median, 'frames')
t0 = time.time(); r = engine.speak('Hello, this is a test of the demo.')
print(f'speak: {time.time() - t0:.1f}s | {r["seconds"]:.2f}s of speech | {len(r["table"])} tokens')
assert len(r['wav']) > 0 and (r['table'].duration_frames >= 1).all()
assert abs(len(r['wav']) - int(r['table'].duration_frames.sum()) * 256) <= 3072, 'audio length should closely equal frames x hop'
r2 = engine.speak_from_table(r['table']); assert len(r2['wav']) == len(r['wav']), 'table round-trip must reproduce the same length'
cmp_ = engine.compare('The quick brown fox jumps over the lazy dog.')
print({k: round(v['seconds'], 2) for k, v in cmp_.items()})
try: engine.speak_from_table(r['table'].assign(phone='XYZ'))
except ValueError as e: print('bad phone is rejected:', e)
r['table'].head(10)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/drive/MyDrive/phonetic-tts/pretrained/hifigan/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch generator.ckpt: Using symlink found at '/content/drive/MyDrive/phonetic-tts/pretrained/hifigan/generator.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: generator


device cpu | checkpoint step 8000 | global median duration 7.0 frames
speak: 2.8s | 2.40s of speech | 25 tokens
{'model': 3.1, 'phone_mean': 2.75, 'global_mean': 2.65}
bad phone is rejected: Unknown phone: XYZ


,phone,duration_frames
0,sil,3
1,HH,4
2,AH0,4
3,L,11
4,OW1,30
5,sil,15
6,DH,4
7,IH1,7
8,S,9
9,IH1,6


## Step 2: Launch the app
Run the next cell, then click the **public URL** it prints (ending in `gradio.live`). The link works while this cell is running (about 72 hours at most). Keep the Colab tab open during the viva, and **test the link on your phone or another laptop beforehand**.

In [10]:
demo = build_demo(engine)
demo.queue().launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://959948c789b7b7a462.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/gradio/processing_utils.py:817: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
/usr/local/lib/python3.13/dist-packages/gradio/processing_utils.py:817: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
/usr/local/lib/python3.13/dist-packages/gradio/processing_utils.py:817: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
/usr/local/lib/python3.13/dist-packages/gradio/processing_utils.py:817: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
/usr/local/lib/python3.13/dist-packages/gradio/processing_utils.py:817: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(dat

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://959948c789b7b7a462.gradio.live


## Demo script for the judges (about 3 minutes)
1. **Speak tab.** Type the first example and press *Generate*. Say: "Text becomes phonemes, then **explicit durations**, then a mel spectrogram, then audio through a pretrained vocoder."
2. **Point at the plots.** The bar chart is the duration of every phoneme (our Phase 1 topic); the spectrogram shows the phoneme boundaries lining up with the sound.
3. **Controls.** Move *Duration scale* to 1.3 and generate again: slower speech with the same voice. Say: "Explicit durations give direct control, which attention-based models do not."
4. **Edit a duration.** In the table, double a vowel's duration (for example change 8 to 20) and press *Re-synthesize*. Say: "We changed one number and that vowel is longer."
5. **Compare tab.** Press *Compare* on the second sentence and play the three versions. Say: "Same model; only the durations differ. Phase 1 predictors improve the timing; our test-set numbers are in the report."
6. **Be honest.** Mention one limit: single speaker, small model, no pitch control; the Phase 1 BiLSTM is evaluated offline on the test set.

**Likely questions:** *Why durations as input?* (control and stability); *What did you train?* (the acoustic model, not the vocoder); *Why does the baseline sound similar?* (the effect on the mel is small, about 2%, see the paired results).

## Step 3 (for the future): save a standalone launcher and hosting notes

In [11]:
APP_PY = r'''"""Standalone launcher (for a Hugging Face Space or a laptop). Paths can be overridden with environment variables."""
import os
from src.app.engine import TTSEngine
from src.app.gradio_app import build_demo

ROOT = os.path.dirname(os.path.abspath(__file__))
engine = TTSEngine(os.environ.get('PGSG_PROCESSED', f'{ROOT}/data/processed'),
                   os.environ.get('PGSG_CKPT', f'{ROOT}/checkpoints/acoustic/best.pt'),
                   os.environ.get('PGSG_HIFIGAN', f'{ROOT}/pretrained/hifigan'))
demo = build_demo(engine)
if __name__ == '__main__':
    demo.launch()
'''

In [12]:
REQ = r'''torch
numpy
pandas
matplotlib
gradio
speechbrain
g2p_en
librosa
soundfile
'''

In [13]:
README_MD = r'''# Demo app: running and hosting

## In Colab (what we use for the viva)
Run notebook 17 top to bottom. The last cell prints a public link (valid while the cell runs).

## Locally
1. `pip install -r requirements_demo.txt`
2. Folder layout next to `app.py`:
   - `src/` (the repo's `src` folder)
   - `data/processed/` with `phase2_contract.json`, `sil_stats.json`, `mel_stats.json`, `phase2_manifest.csv`
   - `checkpoints/acoustic/best.pt`
3. `python app.py` and open the printed address. (HiFi-GAN downloads on first run into `pretrained/hifigan`.)

## Permanent link (free Hugging Face Space)
1. Create a Space (SDK: Gradio, hardware: CPU basic) on huggingface.co.
2. Upload: `app.py`, `requirements.txt` (copy of `requirements_demo.txt`), `src/`, the four files from `data/processed/`, and `checkpoints/acoustic/best.pt` (about 35 MB).
3. The Space builds and serves the same app. CPU speech takes a few seconds per sentence.

## Ideas for later
- Drive the durations with the Phase 1 BiLSTM (expose the Phase 1 feature builder as a function and add it to the `SOURCES` list in `engine.py`).
- Add pitch and energy predictors (full FastSpeech 2) and sliders for them.
- Add a second speaker or a speaking-rate control learned from data.
'''

In [14]:
open(os.path.join(PROJECT_ROOT, 'app.py'), 'w').write(APP_PY)
open(os.path.join(PROJECT_ROOT, 'requirements_demo.txt'), 'w').write(REQ)
open(os.path.join(PROJECT_ROOT, 'README_DEMO.md'), 'w').write(README_MD)
print('wrote app.py, requirements_demo.txt, README_DEMO.md in', PROJECT_ROOT)

wrote app.py, requirements_demo.txt, README_DEMO.md in /content/drive/MyDrive/phonetic-tts
